# Module 18: Training with Transformers (Detailed)


## 🏋️ Fine-Tuning a Transformer

If you want the ultimate state-of-the-art accuracy for your custom NER or TextCat model, you must Fine-Tune a Transformer. 
This involves taking a pre-trained HuggingFace model (which understands English perfectly) and slightly adjusting its weights based on your `train.spacy` dataset.

### 1. Generating a Transformer Config
To switch from a CNN to a Transformer, you simply change the `--optimize` flag when generating `config.cfg`.

```bash
python -m spacy init config config_trf.cfg --lang en --pipeline ner --optimize accuracy
```

If you open `config_trf.cfg`, the `[components.tok2vec]` block is gone, replaced by `[components.transformer]`!


<br><br>

---

<br><br>


### 2. Using Custom HuggingFace Models

You are not locked into RoBERTa! You can inject **ANY** model from the HuggingFace Hub directly into spaCy.

In `config_trf.cfg`, locate this block:
```ini
[components.transformer.model]
@architectures = "spacy-transformers.TransformerModel.v3"
name = "roberta-base"
```

You can change `name` to anything:
- `name = "distilbert-base-uncased"` (For extremely fast, lightweight training)
- `name = "emilyalsentzer/Bio_ClinicalBERT"` (For medical/hospital text)
- `name = "ProsusAI/finbert"` (For financial text)

When you run `spacy train`, spaCy will automatically download those weights from the internet!


<br><br>

---

<br><br>


### 3. GPU Memory Optimization (Critical for Training)

Transformers are massive. If you train a Transformer on a standard 8GB or 12GB GPU, you will almost certainly get a `CUDA Out Of Memory (OOM)` error if you use default settings.

Here are the three ways you configure spaCy to train on consumer hardware:

#### A. Freezing the Transformer
If you freeze the base model, you only train the tiny NER head sitting on top. This cuts VRAM usage by over 50% and speeds up training massively.
```ini
[training]
frozen_components = ["transformer"]
```

#### B. Mixed Precision (AMP)
Deep learning usually uses 32-bit floats. Modern GPUs support 16-bit math, which uses half the memory with almost zero loss in accuracy.
```ini
[training]
use_amp = true
```

#### C. Gradient Accumulation
Transformers need small batch sizes (e.g., `batch_size = 2`) to fit in memory. But neural networks learn terribly with small batches because the gradients are too noisy. 
**Gradient Accumulation** solves this: it runs a batch of 2, saves the math, runs another batch of 2, saves the math, and does this `N` times before updating the weights. If `accumulate_gradient = 4`, it simulates a batch size of 8 without blowing up your VRAM!
```ini
[training]
accumulate_gradient = 4
```


<br><br>

---

<br><br>


### 4. Running the Transformer Training Loop

Transformer training requires a GPU. In your terminal, you must prepend the training command with `-g 0` (telling spaCy to use GPU ID 0).

```bash
python -m spacy train config_trf.cfg --output ./trf_models --paths.train ./train.spacy --paths.dev ./dev.spacy -g 0
```

If you don't use `-g 0`, it will attempt to train the Transformer on your CPU, which could take weeks instead of hours!


<br><br>

---

<br><br>


## 🎉 Summary of Part 6

You have now mastered integrating HuggingFace Transformers into spaCy.
- You know how to load `en_core_web_trf` and access contextualized embeddings.
- You know how to configure `config.cfg` to download ANY model from the HuggingFace Hub.
- You know how to prevent GPU memory crashes using Freezing, Mixed Precision, and Gradient Accumulation.

Next up is the final part of our curriculum: **Part 7: Project & Workflow Management**, where we learn how to wrap our code, datasets, and configurations into automated, shareable spaCy Projects!
